# Lab 1 — LLM Data Pipeline

Build a minimal end-to-end text preprocessing pipeline for language model training: install deps, pull a small dataset, tokenize, chunk, and batch with a collator.

In [19]:
# Install runtime deps (quiet to keep logs short)

!pip install -q transformers datasets torch


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [20]:
import os
import random
from itertools import chain
from collections import Counter

from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
import torch

In [21]:

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DATASET_NAME = "imdb"
DATASET_CONFIG = None
SAMPLE_SIZE = 8000  
BLOCK_SIZE = 192
BATCH_SIZE = 8

print(f"Using {DATASET_NAME} sample={SAMPLE_SIZE}, block size={BLOCK_SIZE}, batch size={BATCH_SIZE}")

Using imdb sample=8000, block size=192, batch size=8


In [22]:
# Pull a small slice of IMDB reviews
raw_ds = load_dataset(DATASET_NAME, DATASET_CONFIG, split="train")
raw_ds = raw_ds.shuffle(seed=SEED).select(range(SAMPLE_SIZE))

print(f"Loaded rows: {len(raw_ds)} | columns: {raw_ds.column_names}")
print(raw_ds)
print("Example review:\n", raw_ds[0]["text"][:200])

Loaded rows: 8000 | columns: ['text', 'label']
Dataset({
    features: ['text', 'label'],
    num_rows: 8000
})
Example review:
 There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. F


In [23]:
# EDA
text_lengths = [len(t) for t in raw_ds["text"]]
label_counts = Counter(raw_ds["label"])

print("Length stats (chars) -> min/mean/max:", min(text_lengths), sum(text_lengths)/len(text_lengths), max(text_lengths))
print("Label balance:", {k: v for k, v in sorted(label_counts.items())})
print("Sample snippet:", raw_ds[0]["text"][:120].replace("\n", " "))

Length stats (chars) -> min/mean/max: 70 1320.085125 10321
Label balance: {0: 3988, 1: 4012}
Sample snippet: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. 


In [24]:
# Tokenizer setup (RoBERTa-base)

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

tokenizer.pad_token = tokenizer.eos_token



def tokenize_batch(batch):

    clean_texts = [(t or "").strip() for t in batch["text"]]

    encoded = tokenizer(

        clean_texts,

        add_special_tokens=True,

        padding=False,

        truncation=False,

        return_attention_mask=True,

        return_special_tokens_mask=False,

    )

    return {

        "input_ids": encoded["input_ids"],

        "attention_mask": encoded["attention_mask"],

    }



tokenized_ds = raw_ds.map(tokenize_batch, batched=True, remove_columns=["text", "label"])

print(f"Tokenized rows: {len(tokenized_ds)} | columns: {tokenized_ds.column_names}")

print("First tokenized sample lens (ids/mask):", len(tokenized_ds[0]["input_ids"]), len(tokenized_ds[0]["attention_mask"]))

Tokenized rows: 8000 | columns: ['input_ids', 'attention_mask']
First tokenized sample lens (ids/mask): 166 166


In [25]:
# Pack tokens into fixed-length blocks

def group_texts(batch):

    flat_ids = list(chain.from_iterable(batch["input_ids"]))

    flat_masks = list(chain.from_iterable(batch["attention_mask"]))



    usable_len = (len(flat_ids) // BLOCK_SIZE) * BLOCK_SIZE

    if usable_len == 0:

        return {"input_ids": [], "attention_mask": []}



    trimmed_ids = flat_ids[:usable_len]

    trimmed_masks = flat_masks[:usable_len]



    def chunk(seq):

        return [seq[i : i + BLOCK_SIZE] for i in range(0, len(seq), BLOCK_SIZE)]



    return {

        "input_ids": chunk(trimmed_ids),

        "attention_mask": chunk(trimmed_masks),

    }



lm_ds = tokenized_ds.map(group_texts, batched=True, batch_size=500)

print(f"Grouped sequences: {len(lm_ds)} | first block lens: {len(lm_ds[0]['input_ids'])}, {len(lm_ds[0]['attention_mask'])}")

Grouped sequences: 12516 | first block lens: 192, 192


In [26]:
def collate_fn(batch):

    ids = torch.stack([torch.as_tensor(item["input_ids"], dtype=torch.long) for item in batch])

    masks = torch.stack([torch.as_tensor(item["attention_mask"], dtype=torch.long) for item in batch])

    labels = ids.clone()

    return {"input_ids": ids, "attention_mask": masks, "labels": labels}



train_loader = DataLoader(lm_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

print(f"Batches per epoch: {len(train_loader)}")

Batches per epoch: 1565


In [27]:
# Peek at one batch

batch = next(iter(train_loader))

print("input_ids shape:", batch["input_ids"].shape)

print("labels shape:", batch["labels"].shape)

print("First row (truncated):", batch["input_ids"][0][:20])

input_ids shape: torch.Size([8, 192])
labels shape: torch.Size([8, 192])
First row (truncated): tensor([   95,   141,  9515, 40485,    51,   269,    58,   328, 41552,  3809,
         1589, 49007,  3809, 48709,  7912, 15176, 18024,    16,     5,  2136])
